# Data Catalog for Scenario Presets

This notebook documents the shared data library that powers the simulator, dashboard, and guided case studies.

It focuses on four questions:

1. Which restaurant layouts and queue presets are available?
2. Which policy and arrival presets are available?
3. Which official case studies exist?
4. Which shared presets does each case study version reference?


## Setup And Data Loading

This section locates the repository root, exposes the simulation package to the notebook kernel, and loads the shared preset library plus the official case study manifests.

Install notebook-specific dependencies with `python3 -m pip install -r notebook/requirements.txt` before running the notebook. If you want to analyse a different scenario library, set `SIM_DATA_ROOT` before starting the kernel so the notebook and API read the same data root.


In [13]:
from __future__ import annotations

from collections import Counter
from html import escape
from pathlib import Path
import sys

from IPython.display import HTML, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "code" / "restaurant_simulation").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from this notebook.")


def table_column_style(column: str) -> str:
    if column in {"description", "message"}:
        return "width: 34%; min-width: 38ch;"
    if column in {"title", "scenario", "focus", "option_label", "metric"}:
        return "width: 22%; min-width: 24ch;"
    if column in {
        "restaurant_layout",
        "queue_structure",
        "reservation_policy",
        "seating_policy",
        "service_policy",
        "arrival_scenario",
        "bands",
        "capacity_mix",
    }:
        return "width: 20%; min-width: 22ch;"
    if column in {
        "average_wait_time",
        "table_utilization_overall",
        "service_level_within_15_min",
        "max_queue_length",
        "peak_time",
    }:
        return "width: 14ch; min-width: 16ch;"
    if column in {"id", "case_study"}:
        return "width: 11ch; min-width: 11ch;"
    if column in {"pair", "version", "time", "group", "table"}:
        return "width: 9ch; min-width: 9ch;"
    if column.endswith("_enabled") or column in {
        "tables",
        "total_seats",
        "largest_table",
        "default_reset",
        "arrival_groups",
        "max_party_size",
        "reservations",
        "walkins",
        "hold_minutes",
        "peak_waiting",
        "final_waiting",
        "snapshots",
        "groups_served",
        "groups_abandoned",
    }:
        return "width: 12ch; min-width: 12ch;"
    return "width: 14ch; min-width: 14ch;"


def table_cell_style(column: str, *, header: bool = False) -> str:
    styles = [
        "padding: 0.4rem 0.6rem",
        "border: 1px solid #d0d7de",
        "text-align: right",
        "vertical-align: top",
        "overflow-wrap: anywhere",
        "word-break: break-word",
        "line-height: 1.35",
    ]
    if header:
        styles.extend(
            [
                "font-weight: 600",
                "background: #e9eef5",
                "color: #1f2328",
                "white-space: normal",
            ]
        )
    elif column in {"pair", "version", "time", "group", "table"} or column.endswith("_enabled"):
        styles.append("white-space: nowrap")
    return "; ".join(styles)


def render_table(rows: list[dict[str, object]], columns: list[str]) -> HTML:
    colgroup = "".join(f"<col style=\"{table_column_style(column)}\">" for column in columns)
    header = "".join(
        f"<th style=\"{table_cell_style(column, header=True)}\">{escape(column)}</th>"
        for column in columns
    )
    body_rows = []
    for row in rows:
        body_cells = "".join(
            f"<td style=\"{table_cell_style(column)}\">{escape(str(row.get(column, '')))}</td>"
            for column in columns
        )
        body_rows.append(f"<tr>{body_cells}</tr>")
    empty_row = (
        f"<tr><td colspan='{len(columns)}' style=\"{table_cell_style('description')}\">No rows</td></tr>"
    )
    body = "".join(body_rows) or empty_row
    return HTML(
        "<div style=\"overflow-x: auto; margin: 0.5rem 0 1rem;\">"
        "<table style=\"border-collapse: collapse; table-layout: fixed; width: 100%;\">"
        f"<colgroup>{colgroup}</colgroup>"
        f"<thead><tr>{header}</tr></thead>"
        f"<tbody>{body}</tbody>"
        "</table>"
        "</div>"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
CODE_ROOT = REPO_ROOT / "code"

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from restaurant_simulation import discover_case_studies, load_builder_presets, resolve_data_root

DATA_ROOT = resolve_data_root()
presets = load_builder_presets(data_root=DATA_ROOT)
case_studies = discover_case_studies(data_root=DATA_ROOT)

print(f"Repository root: {REPO_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Loaded {len(presets['restaurant_layouts'])} restaurant layouts, {len(presets['queue_structures'])} queue presets, and {len(case_studies)} official case studies.")


Repository root: /Users/grantwright/Desktop/COMP 1110/COMP-1110-C08
Data root: /Users/grantwright/Desktop/COMP 1110/COMP-1110-C08/data
Loaded 8 restaurant layouts, 4 queue presets, and 7 official case studies.


## Restaurant, Queue, And Arrival Presets

This section summarizes the top-level preset library so the reader can quickly scan which real restaurant layouts, abstract queue templates, and arrival patterns are available for reuse. The tables below also surface seat counts and demand-size metadata so readers can judge realism and compatibility at a glance.


In [14]:
def capacity_mix(tables: list[dict[str, object]]) -> str:
    counts = Counter(int(table["capacity"]) for table in tables)
    return ", ".join(f"{size}-seat x{count}" for size, count in sorted(counts.items()))


def seat_count(tables: list[dict[str, object]]) -> int:
    return sum(int(table["capacity"]) for table in tables)


def queue_bands(queue_definitions: list[dict[str, object]]) -> str:
    return ", ".join(
        f"{item['queue_id']}={item['min_size']}-{item['max_size']}" for item in queue_definitions
    )


restaurant_layout_rows = [
    {
        "id": item["id"],
        "title": item["title"],
        "service_model": item["data"].get("optional_operational_defaults", {}).get("service_model", ""),
        "signature_cuisine": item["data"].get("optional_operational_defaults", {}).get("signature_cuisine", ""),
        "tables": len(item["data"]["tables"]),
        "total_seats": seat_count(item["data"]["tables"]),
        "largest_table": max(int(table["capacity"]) for table in item["data"]["tables"]),
        "capacity_mix": capacity_mix(item["data"]["tables"]),
        "default_reset": item["data"]["default_reset_policy"]["default_reset_minutes"],
        "description": item["description"],
    }
    for item in presets["restaurant_layouts"]
]

queue_structure_rows = [
    {
        "id": item["id"],
        "title": item["title"],
        "template_type": "abstract queue template",
        "queue_mode": item["data"]["queue_mode"],
        "bands": queue_bands(item["data"]["queue_definitions"]),
        "description": item["description"],
    }
    for item in presets["queue_structures"]
]

arrival_scenario_rows = [
    {
        "id": item["id"],
        "title": item["title"],
        "arrival_groups": item["row_count"],
        "max_party_size": item["max_group_size"],
        "reservations": item["reservation_groups"],
        "walkins": item["walkin_groups"],
        "description": item["description"],
    }
    for item in presets["arrival_scenarios"]
]

print("Restaurant layouts")
display(
    render_table(
        restaurant_layout_rows,
        [
            "id",
            "title",
            "service_model",
            "signature_cuisine",
            "tables",
            "total_seats",
            "largest_table",
            "capacity_mix",
            "default_reset",
            "description",
        ],
    )
)
print("Queue structures")
display(render_table(queue_structure_rows, ["id", "title", "template_type", "queue_mode", "bands", "description"]))
print("Arrival scenarios")
display(render_table(arrival_scenario_rows, ["id", "title", "arrival_groups", "max_party_size", "reservations", "walkins", "description"]))


Restaurant layouts


id,title,service_model,signature_cuisine,tables,total_seats,largest_table,capacity_mix,default_reset,description
layout_fine_dining,Cedar Tasting Room,fine_dining,seasonal tasting menu,12,48,8,"2-seat x5, 4-seat x3, 6-seat x3, 8-seat x1",5,"Reservation-led fine-dining room with more tables than before, but still intentionally tighter and more capacity-constrained than the casual venues."
layout_burger_fast_casual,Ember Lane Burger Hall,fast_casual,"burgers, fries, and shakes",20,78,8,"2-seat x6, 4-seat x10, 6-seat x3, 8-seat x1",3,"Fast-casual burger hall with many two-top and four-top tables, a few family booths, and only a small amount of flexible larger seating."
layout_dessert_cafe,Moonlight Dessert Studio,dessert_lounge,"cakes, parfaits, and specialty coffee",16,64,8,"2-seat x6, 4-seat x6, 6-seat x2, 8-seat x2",3,"Dessert cafe built around pairs and small friend groups, with only a few larger tables reserved for celebrations or shared platters."
layout_dim_sum_hall,Pearl River Dim Sum Hall,cart_service,Cantonese dim sum and banquet dining,24,132,8,"2-seat x2, 4-seat x8, 6-seat x8, 8-seat x6",5,"Dim sum hall with many medium and large banquet tables, but still enough variation that queue resolution matters during heavy family demand."
layout_family_trattoria,Piazza Family Trattoria,casual_full_service,Italian family dining,24,116,8,"2-seat x4, 4-seat x10, 6-seat x6, 8-seat x4",4,"Family trattoria with a deep mix of four-tops, several six-top family tables, and multiple larger tables for celebrations."
layout_korean_bbq,Seoul Fire BBQ House,interactive_grill_service,Korean barbecue,20,112,8,"4-seat x8, 6-seat x8, 8-seat x4",6,"Large Korean BBQ room where grill tables turn slowly, so reset speed and cleaning capacity still shape throughput even with a larger floor."
layout_ramen_bar,Shoyu Ramen Counter,fast_full_service,Japanese ramen and small plates,18,36,6,"1-seat x10, 2-seat x4, 4-seat x3, 6-seat x1",4,"Counter-led ramen shop with many solo stools, several small pair tables, and only a few larger tables toward the back of the room."
layout_bubble_tea_express,Tea Cloud Bubble Bar,counter_service,bubble tea and light snacks,14,28,6,"1-seat x6, 2-seat x6, 4-seat x1, 6-seat x1",2,"Compact bubble-tea shop with a long counter, several window stools, a few short-stay tables, and one larger shared table for small groups."


Queue structures


id,title,template_type,queue_mode,bands,description
queue_balanced_size,Queue template: balanced size-based waitlists,abstract queue template,size_based,"Q1=1-2, Q2=3-4, Q3=5-6, Q4=7-20",Abstract queue template for the builder: parties are split into broad size bands to improve table matching.
queue_coarse_party_bands,Queue template: coarse party-size bands,abstract queue template,size_based,"Q_SMALL=1-2, Q_MEDIUM=3-6, Q_LARGE=7-20",Abstract queue template for the builder: three broad queue buckets trade precision for lower host workload.
queue_fine_party_bands,Queue template: fine-grained party-size bands,abstract queue template,size_based,"Q1=1-1, Q2=2-2, Q3=3-4, Q4=5-6, Q5=7-20",Abstract queue template for the builder: more queue buckets create tighter matching but demand more attention.
queue_single_line,Queue template: single waitlist for all parties,abstract queue template,single,Q_ALL=1-20,Abstract queue template for the builder: everyone waits in one visible line regardless of party size.


Arrival scenarios


id,title,arrival_groups,max_party_size,reservations,walkins,description
dinner_peak_base,Dinner Peak,30,6,4,26,"30 arrival groups, up to 6 guests, 4 reservations"
large_party_heavy_base,Large Party Heavy,36,8,11,25,"36 arrival groups, up to 8 guests, 11 reservations"
lunch_rush_base,Lunch Rush,28,6,5,23,"28 arrival groups, up to 6 guests, 5 reservations"
reservation_mixed_base,Reservation Mixed,34,6,19,15,"34 arrival groups, up to 6 guests, 19 reservations"
stress_peak_base,Stress Peak,28,6,0,28,"28 arrival groups, up to 6 guests, walk-in only"


## Policy Library Overview

This section lists the reusable seating, service, and reservation-control presets that can be combined inside the official case study manifests.


In [15]:
policy_tables = {
    "seating_policies": [
        {
            "id": item["id"],
            "title": item["title"],
            "queue_rule": item["data"]["queue_rule"],
            "selection_rule": item["data"]["selection_rule"],
            "reservation_priority": item["data"]["reservation_priority"],
        }
        for item in presets["seating_policies"]
    ],
    "service_policies": [
        {
            "id": item["id"],
            "title": item["title"],
            "seating_actions": item["data"]["max_seating_actions_per_event_time"],
            "cleaning_actions": item["data"]["max_cleaning_actions_per_event_time"],
            "kitchen_load_mode": item["data"]["kitchen_load_mode"],
            "abandonment_enabled": item["data"]["abandonment_enabled"],
        }
        for item in presets["service_policies"]
    ],
    "reservation_policies": [
        {
            "id": item["id"],
            "title": item["title"],
            "reservation_system_enabled": item["data"]["reservation_system_enabled"],
            "hold_tables_for_reservations": item["data"]["hold_tables_for_reservations"],
            "default_hold_minutes": item["data"]["default_hold_minutes"],
        }
        for item in presets["reservation_policies"]
    ],
}

policy_columns = {
    "seating_policies": ["id", "title", "queue_rule", "selection_rule", "reservation_priority"],
    "service_policies": ["id", "title", "seating_actions", "cleaning_actions", "kitchen_load_mode", "abandonment_enabled"],
    "reservation_policies": ["id", "title", "reservation_system_enabled", "hold_tables_for_reservations", "default_hold_minutes"],
}

for label, rows in policy_tables.items():
    print(label.replace("_", " ").title())
    display(render_table(rows, policy_columns[label]))


Seating Policies


id,title,queue_rule,selection_rule,reservation_priority
seating_best_fit,Best-Fit Seating,fcfs_within_queue,best_fit,False
seating_fcfs,FCFS Earliest Suitable,fcfs_within_queue,earliest_suitable,False
seating_reservation_priority,Reservation Priority Seating,fcfs_within_queue,earliest_suitable,True


Service Policies


id,title,seating_actions,cleaning_actions,kitchen_load_mode,abandonment_enabled
service_default,Default Service Capacity,2,2,moderate_peak,False
service_high_cleaning_capacity,High Cleaning Capacity,3,3,light_peak,False
service_low_cleaning_capacity,Low Cleaning Capacity,2,1,moderate_peak,False


Reservation Policies


id,title,reservation_system_enabled,hold_tables_for_reservations,default_hold_minutes
reservation_disabled,No Reservation Hold,False,False,0
reservation_enabled,Reservation Hold Enabled,True,True,10


## Official Case Study Mapping

This section expands each official pair into its A and B starter definitions so the reader can see exactly which shared presets are referenced by each guided comparison.


In [16]:
lookup = {
    key: {item["id"]: item["title"] for item in values}
    for key, values in presets.items()
}

case_study_rows = []
for case in case_studies:
    for version, starter in sorted(case.starter_versions.items()):
        case_study_rows.append(
            {
                "case_study": case.case_study,
                "title": case.title,
                "version": version,
                "focus": case.focus_label,
                "option_label": starter.label,
                "restaurant_layout": lookup["restaurant_layouts"][starter.restaurant_layout_id],
                "queue_structure": lookup["queue_structures"][starter.queue_structure_id],
                "reservation_policy": lookup["reservation_policies"][starter.reservation_policy_id],
                "seating_policy": lookup["seating_policies"][starter.seating_policy_id],
                "service_policy": lookup["service_policies"][starter.service_policy_id],
                "arrival_scenario": lookup["arrival_scenarios"][starter.arrival_scenario_id],
                "hold_minutes": starter.hold_minutes,
                "abandonment_enabled": starter.abandonment_enabled,
            }
        )

display(
    render_table(
        case_study_rows,
        [
            "case_study",
            "title",
            "version",
            "focus",
            "option_label",
            "restaurant_layout",
            "queue_structure",
            "reservation_policy",
            "seating_policy",
            "service_policy",
            "arrival_scenario",
            "hold_minutes",
            "abandonment_enabled",
        ],
    )
)


case_study,title,version,focus,option_label,restaurant_layout,queue_structure,reservation_policy,seating_policy,service_policy,arrival_scenario,hold_minutes,abandonment_enabled
pair_01_table_mix,Pair 01: Dessert Lounge vs Family Trattoria,A,Dining room footprint,Dessert lounge layout,Moonlight Dessert Studio,Queue template: balanced size-based waitlists,Reservation Hold Enabled,Best-Fit Seating,Default Service Capacity,Large Party Heavy,10,False
pair_01_table_mix,Pair 01: Dessert Lounge vs Family Trattoria,B,Dining room footprint,Family trattoria layout,Piazza Family Trattoria,Queue template: balanced size-based waitlists,Reservation Hold Enabled,Best-Fit Seating,Default Service Capacity,Large Party Heavy,10,False
pair_02_single_vs_multi_queue,Pair 02: Single Queue vs Multi Queue,A,Waitlist structure,Single shared line,Shoyu Ramen Counter,Queue template: single waitlist for all parties,No Reservation Hold,FCFS Earliest Suitable,Default Service Capacity,Lunch Rush,0,False
pair_02_single_vs_multi_queue,Pair 02: Single Queue vs Multi Queue,B,Waitlist structure,Split size-based lines,Shoyu Ramen Counter,Queue template: balanced size-based waitlists,No Reservation Hold,FCFS Earliest Suitable,Default Service Capacity,Lunch Rush,0,False
pair_03_coarse_vs_fine_queue,Pair 03: Coarse vs Fine Queue Buckets,A,Queue resolution,Broader queue bands,Pearl River Dim Sum Hall,Queue template: coarse party-size bands,Reservation Hold Enabled,FCFS Earliest Suitable,Default Service Capacity,Large Party Heavy,10,False
pair_03_coarse_vs_fine_queue,Pair 03: Coarse vs Fine Queue Buckets,B,Queue resolution,Fine-grained queue bands,Pearl River Dim Sum Hall,Queue template: fine-grained party-size bands,Reservation Hold Enabled,FCFS Earliest Suitable,Default Service Capacity,Large Party Heavy,10,False
pair_04_no_reservation_hold_vs_hold,Pair 04: No Reservation Hold vs Reservation Hold,A,Reservation protection,No hold after booking time,Cedar Tasting Room,Queue template: balanced size-based waitlists,No Reservation Hold,Reservation Priority Seating,Default Service Capacity,Reservation Mixed,0,False
pair_04_no_reservation_hold_vs_hold,Pair 04: No Reservation Hold vs Reservation Hold,B,Reservation protection,Protect bookings through grace period,Cedar Tasting Room,Queue template: balanced size-based waitlists,Reservation Hold Enabled,Reservation Priority Seating,Default Service Capacity,Reservation Mixed,15,False
pair_05_low_vs_high_cleaning_capacity,Pair 05: Low vs High Cleaning Capacity,A,Reset capacity,Tight cleaning bottleneck,Seoul Fire BBQ House,Queue template: balanced size-based waitlists,Reservation Hold Enabled,Best-Fit Seating,Low Cleaning Capacity,Dinner Peak,10,False
pair_05_low_vs_high_cleaning_capacity,Pair 05: Low vs High Cleaning Capacity,B,Reset capacity,Expanded cleaning team,Seoul Fire BBQ House,Queue template: balanced size-based waitlists,Reservation Hold Enabled,Best-Fit Seating,High Cleaning Capacity,Dinner Peak,10,False
